In [1]:

from azure.ai.ml import MLClient, command, Input
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AzureBlobDatastore, Data
from azure.ai.ml.constants import AssetTypes
from azure.storage.blob import BlobServiceClient

import json
import os

from dotenv import load_dotenv

load_dotenv()

CONFIG_FILE_PATH = "./infra/infra_config.json"
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")  # run: az account show --query id --output tsv

# Load the configuration from the JSON file
with open(CONFIG_FILE_PATH, "r") as f:
    config = json.load(f)

In [2]:

# 1. Configuration Constants (Match your Azure/Terraform setup)
RESOURCE_GROUP = config["resource_group"]
WORKSPACE_NAME = config["workspace_name"]
COMPUTE_NAME   = config["compute_name"]

STORAGE_ACCOUNT_NAME = config["storage_account_name"]  # Matches your Terraform storage account name
STORAGE_CONTAINER_NAME = config["container_name"]  # Matches your Terraform container name

STORAGE_ACCOUNT_URI = f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net"

LOCAL_DATA_PATH = "C:\\Users\\m_kal\\Downloads\\datasets\\rossmann-store-sales"  # Local folder to upload (ensure it exists)
DESTINATION_BLOB_PATH = "rossmann/"  # Destination path inside the container

DATASTORE_NAME = "data_container_datastore"  # Name for the Datastore that points to the custom container
DATASTORE_URI = f"azureml://datastores/{DATASTORE_NAME}/paths/{DESTINATION_BLOB_PATH}"  # Path to the uploaded data in the datastore


In [3]:

# 1. Authenticate to Azure ML (Moved up so it exists before we try to use it)
print("Connecting to Azure ML Workspace...")
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)


Connecting to Azure ML Workspace...


In [4]:

# 2. Register your custom Terraform container as a new Datastore
print("Registering custom datastore...")
custom_datastore = AzureBlobDatastore(
    name=DATASTORE_NAME,
    description="Datastore pointing to our custom data_container container",
    account_name=STORAGE_ACCOUNT_NAME,
    container_name=STORAGE_CONTAINER_NAME # Matches your Terraform container name!
)
ml_client.datastores.create_or_update(custom_datastore)


Registering custom datastore...


AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'data_container_datastore', 'description': 'Datastore pointing to our custom data_container container', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/dc6fd0ed-8e9d-4a63-b4a6-cc23be43b154/resourceGroups/rg-ml-workspace-rossmann/providers/Microsoft.MachineLearningServices/workspaces/mlw-workspace/datastores/data_container_datastore', 'Resource__source_path': '', 'base_path': 'c:\\Users\\m_kal\\Downloads\\rossmann_store_sales', 'creation_context': None, 'credentials': <azure.ai.ml.entities._credentials.NoneCredentialConfiguration object at 0x000001E0467002D0>, 'container_name': 'datacontainer1233', 'account_name': 'stmlws1233', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [ ]:

# 3. Upload local data and register it as a Data asset on the custom datastore.
#    Step 1: Upload the local folder to the custom container via the Azure Storage SDK.
#    (Assumes the local ./data folder exists and azure-storage-blob is installed)

data_asset_name = "my_uploaded_dataset"

print(f"Uploading local directory '{LOCAL_DATA_PATH}' to custom datastore container...")


# ==============================================================================
# WHY WE CHUNK AND LIMIT SINGLE PUT SIZE:
# 
# 1. THE 64MB DEFAULT BOTTLENECK:
#    By default, the Azure Python SDK uploads any file under 64MB as a single,
#    unbroken HTTP PUT request. 
# 
# 2. THE TIMEOUT ISSUE:
#    For a 37MB file on a slower or firewalled/VPN connection, a single-stream
#    upload takes too long. If there is even a micro-hiccup in the connection, 
#    the stream stalls, triggers a network timeout (usually at 2 minutes), 
#    and the entire upload fails.
# 
# 3. HOW CHUNKING FIXES IT:
#    By setting `max_single_put_size` to 4MB, we force the SDK to slice any 
#    file larger than 4MB into smaller blocks (e.g., nine 4MB blocks for 37MB).
#    
#    - Reliability: If one 4MB chunk fails due to a network hiccup, the SDK 
#      only has to retry that specific 4MB chunk, not start the whole 37MB over.
#    - Speed: Using `max_concurrency=4` allows the SDK to upload up to 4 of 
#      these chunks simultaneously, significantly speeding up the transfer.
# ==============================================================================


blob_service = BlobServiceClient(
    account_url=STORAGE_ACCOUNT_URI,
    credential=DefaultAzureCredential(),
    # If file is larger than 4MB, break it up
    max_single_put_size=4 * 1024 * 1024,  
    # Upload chunks in 4MB pieces
    max_block_size=4 * 1024 * 1024        
)


container_client = blob_service.get_container_client(STORAGE_CONTAINER_NAME)

for root, _, files in os.walk(LOCAL_DATA_PATH):
    for file in files:
        print(f"Uploading {file}...")
        local_file_path = os.path.join(root, file)
        blob_rel_path = os.path.relpath(local_file_path, LOCAL_DATA_PATH).replace("\\", "/")
        blob_name = DESTINATION_BLOB_PATH + blob_rel_path
        
        # Get the individual blob client
        blob_client = container_client.get_blob_client(blob_name)
        
        with open(local_file_path, "rb") as data:
            blob_client.upload_blob(
                data, 
                overwrite=True,
                max_concurrency=4,        # Upload chunks in parallel
                timeout=300               # Timeout limit for the entire operation (in seconds)
            )

print("Upload complete.")

Uploading local directory 'C:\Users\m_kal\Downloads\datasets\rossmann-store-sales' to custom datastore container...
Uploading sample_submission.csv...
Uploading store.csv...
Uploading test.csv...
Uploading train.csv...
Upload complete.


In [6]:

#    Step 2: Register the cloud path as a Data asset using the azureml:// URI.
my_data_asset = Data(
    name=data_asset_name,
    version="1.0.0",
    description="Dataset uploaded to the custom data_container_datastore",
    path=DATASTORE_URI,
    type=AssetTypes.URI_FOLDER
)
registered_data_asset = ml_client.data.create_or_update(my_data_asset)
print(f"Data asset registered as: {registered_data_asset.name}")


Data asset registered as: my_uploaded_dataset


In [ ]:
# 4. Define the Command Job configuration
job = command(
    display_name="Generic ML Pipeline Run",
    experiment_name="generic-jobs",
    description="A generic template for running code on Azure Spot ML Clusters",
    
    # Point to local directories and define execution script
    code="./src",  # Packs everything inside ./src (including main.py and config.yaml)
    command="python main.py --config debug_config.yaml --data_dir ${{inputs.raw_data}}",
    compute=COMPUTE_NAME,
    
    # The software environment to run inside the container
    # List of curated environments: https://ml.azure.com/registries/azureml/environments
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    
    # Reference the freshly registered cloud dataset directly!
    inputs={
        "raw_data": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{data_asset_name}:1.0.0", 
            mode="ro_mount"
        ),
    },
    
    # Guardrail timeout (e.g., 3600 seconds = 1 hour limit)
    limits={"timeout": 3600} 
)


In [ ]:

# 5. Submit to Azure ML
print("Submitting command job...")
returned_job = ml_client.jobs.create_or_update(job)

print("Job Submitted successfully!")
print(f"Job Name:   {returned_job.name}")
print(f"Status:     {returned_job.status}")
print(f"Studio Link: {returned_job.studio_url}")